# Hybrid retrieval demo

Query the Utah FORGE report corpus with **BM25**, **dense (Chroma + BGE)**, or **hybrid RRF** retrieval.

Run `scripts/build_index.py` first if indexes are missing.

In [1]:
from __future__ import annotations

from pathlib import Path

from IPython.display import Markdown, display

from wellground.retrieval import (
    BM25Index,
    DenseIndex,
    HybridRetriever,
)

PROJECT_ROOT = Path("..").resolve()
BM25_DIR = PROJECT_ROOT / "data/processed/bm25"
CHROMA_DIR = PROJECT_ROOT / "data/processed/chroma"
TOP_K = 6

bm25 = BM25Index.load(BM25_DIR)
dense = DenseIndex.load(CHROMA_DIR)
retriever = HybridRetriever(bm25, dense)

print(f"BM25 chunks: {len(bm25.chunks)}")
print(f"Dense chunks: {dense.chunk_count}")

/Users/mehuljain/Documents/Projects/well/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7171.66it/s]


BM25 chunks: 249
Dense chunks: 249


In [2]:
def show_hits(title: str, hits, *, score_fmt: str = ".3f") -> None:
    display(Markdown(f"### {title}"))
    if not hits:
        display(Markdown("_No hits_"))
        return

    lines: list[str] = []
    for hit in hits:
        preview = hit.chunk.text.replace("\n", " ").strip()[:180]
        extra = ""
        if hasattr(hit, "bm25_rank"):
            extra = f" | bm25 #{hit.bm25_rank or '-'} | dense #{hit.dense_rank or '-'}"
        lines.append(
            f"**#{hit.rank}** `{hit.chunk.chunk_id}` "
            f"(score={hit.score:{score_fmt}}{extra})  \n"
            f"{preview}"
        )
    display(Markdown("\n\n".join(lines)))


def compare_query(query: str, *, top_k: int = TOP_K) -> None:
    display(Markdown(f"## Query: `{query}`"))
    show_hits("BM25", bm25.search(query, top_k=top_k))
    show_hits("Dense (BGE + Chroma)", dense.search(query, top_k=top_k))
    show_hits("Hybrid (RRF)", retriever.search(query, top_k=top_k), score_fmt=".4f")

In [3]:
# Conceptual / paraphrase-style question
compare_query("What issues were reported on 16A during the circulation test?")

## Query: `What issues were reported on 16A during the circulation test?`

### BM25

**#1** `inj_prod/16A/University of Utah_Forge16A(78)-32_SLB_17Aug2024_Injection Profile Final Report#p004#c00` (score=14.697)  
University of Utah Forge 16A(78)-32 Interpretation Remarks SLB University of Utah / Forge 16A(78)-32 / Injection Profile / Final Report llk 4/15 This interpretation is based on Ult

**#2** `inj_prod/16B/University of Utah_Forge16B(78)-32_SLB_28Aug2024_DiDrill_Production Log Final Report#p004#c00` (score=14.437)  
University of Utah Forge 16B(78)-32 Interpretation Remarks SLB University of Utah / Forge 16B(78)-32 / Producing Profile / Final Report llk 4/14 This interpretation is based on Hi-

**#3** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT No.1, 8-6-2024#p001#c00` (score=9.082)  
1Report No: Report For 06:00 AM 06-Aug-24 Daily Report Well ID: Utah FORGE 16A(78)-32-STIM1 Well Name: FORGE 16A(78)-32-STIM1 Field: FORGE County: Beaver State: UT Country: United 

**#4** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT11, 8-17-2024#p001#c00` (score=8.707)  
11Report No: Report For 06:00 AM 17-Aug-24 Daily Report Well ID: Utah FORGE FORGE 16B(78)-32 Well Name: FORGE 16B(78)-32 Field: FORGE Sect: 32 Town: 26S Rng: 9W County: BEAVER Stat

**#5** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT14, 8-20-2024#p001#c00` (score=8.302)  
14Report No: Report For 06:00 AM 20-Aug-24 Daily Report Well ID: Utah FORGE FORGE 16B(78)-32 Well Name: FORGE 16B(78)-32 Field: FORGE Sect: 32 Town: 26S Rng: 9W County: BEAVER Stat

**#6** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT13, 8-18-2024#p001#c02` (score=8.230)  
'. Pull up to 9,170' and wait for confirmation that logs were good. 21:00 23:00 2.00 10,987.0 LOG Pull out of hole from 9,170' to 100'. Shut down pumps, pull tool into lubricator, 

### Dense (BGE + Chroma)

**#1** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT4, 8-9-2024#p001#c01` (score=0.743)  
987.0 OTHER Test additional lines, check all flowline valves, shut in 16B(78)-32, change valves on 16A(78)-32 to go downhole and set up sampling ports. 9:00 6:00 21.00 10,987.0 PUM

**#2** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT14, 8-20-2024#p001#c02` (score=0.737)  
30 - P 265 psi, T 357°F, F 334 gpm 03:30 - P 267 psi, T 357°F, F 338 gpm 05:30 - P 267 psi, T 358°F, F 328 gpm Management Summary Remove BOP from 16A and nipple up on 16B. PJSM wit

**#3** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT5, 8-11-2024#p001#c01` (score=0.733)  
80 psi, T 278°F, F 121 gpm 13:00 - P 80 psi, T 282°F, F 122 gpm 13:00 6:00 17.00 10,947.0 EVAL Increase pump rate on 16A(78)-32 to 7.5 bpm, Step 6 of pump schedule. 13:30 - P 82 ps

**#4** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT4, 8-10-2024#p001#c01` (score=0.726)  
Stop pumping on 16A(78)-32 and shut well in. Monitor pressures on 16B(78)-32. 10:00 - 307 psi 12:00 - 280 psi 14:00 - 258 psi 16:00 - 242 psi 17:00 - 236 psi 17:00 1:00 8.00 10,947

**#5** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT11, 8-17-2024#p001#c01` (score=0.725)  
16B. Neubrex reported fiber optics failure @ 03:39, 4,250' MD on 8-16-2024. Alan Reynold reported Baker P/T gauge still getting readings. 07:30 - P 266 psi, T 355°F, F 321 gpm 09:3

**#6** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT12, 8-17-2024#p001#c01` (score=0.724)  
operable and cannot get another crane until morning. SLB assembled PLT tool and SureFire removed nightcap on 16A and bolted up adapter flange for lubricator and assembled lubricato

### Hybrid (RRF)

**#1** `inj_prod/16A/University of Utah_Forge16A(78)-32_SLB_17Aug2024_Injection Profile Final Report#p004#c00` (score=0.0164 | bm25 #1 | dense #-)  
University of Utah Forge 16A(78)-32 Interpretation Remarks SLB University of Utah / Forge 16A(78)-32 / Injection Profile / Final Report llk 4/15 This interpretation is based on Ult

**#2** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT4, 8-9-2024#p001#c01` (score=0.0164 | bm25 #- | dense #1)  
987.0 OTHER Test additional lines, check all flowline valves, shut in 16B(78)-32, change valves on 16A(78)-32 to go downhole and set up sampling ports. 9:00 6:00 21.00 10,987.0 PUM

**#3** `inj_prod/16B/University of Utah_Forge16B(78)-32_SLB_28Aug2024_DiDrill_Production Log Final Report#p004#c00` (score=0.0161 | bm25 #2 | dense #-)  
University of Utah Forge 16B(78)-32 Interpretation Remarks SLB University of Utah / Forge 16B(78)-32 / Producing Profile / Final Report llk 4/14 This interpretation is based on Hi-

**#4** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT14, 8-20-2024#p001#c02` (score=0.0161 | bm25 #- | dense #2)  
30 - P 265 psi, T 357°F, F 334 gpm 03:30 - P 267 psi, T 357°F, F 338 gpm 05:30 - P 267 psi, T 358°F, F 328 gpm Management Summary Remove BOP from 16A and nipple up on 16B. PJSM wit

**#5** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT No.1, 8-6-2024#p001#c00` (score=0.0159 | bm25 #3 | dense #-)  
1Report No: Report For 06:00 AM 06-Aug-24 Daily Report Well ID: Utah FORGE 16A(78)-32-STIM1 Well Name: FORGE 16A(78)-32-STIM1 Field: FORGE County: Beaver State: UT Country: United 

**#6** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT5, 8-11-2024#p001#c01` (score=0.0159 | bm25 #- | dense #3)  
80 psi, T 278°F, F 121 gpm 13:00 - P 80 psi, T 282°F, F 122 gpm 13:00 6:00 17.00 10,947.0 EVAL Increase pump rate on 16A(78)-32 to 7.5 bpm, Step 6 of pump schedule. 13:30 - P 82 ps

In [4]:
# Exact report id / filename-style lookup
compare_query("FORGE 16A(78)-32 Circulation Test RPT10")

## Query: `FORGE 16A(78)-32 Circulation Test RPT10`

### BM25

**#1** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT10, 8-16-2024#p001#c02` (score=9.986)  
.0 10,208.1 8,061.0 INT2 9.500 38 OTHER Bulk InventoryWater / Fluids InventoryWater / Fluids HauledSafety Information First Aid Treatments: 0 0 0Medical Treatments: Lost Time Incid

**#2** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT10, 8-16-2024#p001#c00` (score=8.501)  
10Report No: Report For 06:00 AM 16-Aug-24 Daily Report Well ID: Utah FORGE FORGE 16B(78)-32 Well Name: FORGE 16B(78)-32 Field: FORGE Sect: 32 Town: 26S Rng: 9W County: BEAVER Stat

**#3** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT10, 8-15-2024#p001#c01` (score=8.490)  
11:30 - 2,875 psi 13:30 - 2,866 psi 15:30 - 2,925 psi 17:30 - 2,929 psi 19:30 - 2,944 psi 21:30 - 2,931 psi 23:30 - 2,936 psi 01:30 - 2,935 psi 03:30 - lost communication 06:00 - 2

**#4** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT10, 8-15-2024#p001#c00` (score=8.291)  
10Report No: Report For 06:00 AM 15-Aug-24 Daily Report Well ID: Utah FORGE 16A(78)-32-STIM1 Well Name: FORGE 16A(78)-32-STIM1 Field: FORGE County: Beaver State: UT Country: United

**#5** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT10, 8-16-2024#p001#c01` (score=6.175)  
psi, T 355°F, F 325 gpm 09:30 - P 249 psi, T 354°F, F 325 gpm 11:30 - P 248 psi, T 351°F, F 332 gpm 13:30 - P 248 psi, T 353°F, F 331 gpm 15:30 - P 262 psi, T 352°F, F 328 gpm 17:3

**#6** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT4, 8-9-2024#p001#c00` (score=5.782)  
4Report No: Report For 06:00 AM 09-Aug-24 Daily Report Well ID: Utah FORGE 16A(78)-32-STIM1 Well Name: FORGE 16A(78)-32-STIM1 Field: FORGE County: Beaver State: UT Country: United 

### Dense (BGE + Chroma)

**#1** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT4, 8-9-2024#p001#c01` (score=0.842)  
987.0 OTHER Test additional lines, check all flowline valves, shut in 16B(78)-32, change valves on 16A(78)-32 to go downhole and set up sampling ports. 9:00 6:00 21.00 10,987.0 PUM

**#2** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT5, 8-11-2024#p001#c01` (score=0.831)  
80 psi, T 278°F, F 121 gpm 13:00 - P 80 psi, T 282°F, F 122 gpm 13:00 6:00 17.00 10,947.0 EVAL Increase pump rate on 16A(78)-32 to 7.5 bpm, Step 6 of pump schedule. 13:30 - P 82 ps

**#3** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT3, 8-9-2024#p001#c01` (score=0.830)  
30 Day Circulation Test. 09:15 - 16B wellhead pressure 0 psi, 12:00 - 27 psi, 14:00 - 63 psi, 16:00 - 99 psi, 18:00 - 133 psi, 20:00 - 164 psi, 22:00 - 192 psi, 00:00 - 221 psi, 02

**#4** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT No.1, 8-6-2024#p001#c01` (score=0.827)  
pad and began rigging up. Bulk InventoryWater / Fluids InventoryWater / Fluids HauledSafety Information Meetings/Drills First Aid Treatments: Safety 0 0 0Medical Treatments: Lost T

**#5** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT8, 8-14-2024#p001#c01` (score=0.821)  
8-14-2024. 07:30 - P 228 psi, T 324°F, F 212 gpm 09:30 - P 225 psi, T 311F, F 305 gpm 11:30 - P 223 psi, T 311°F, F 302 gpm 13:30 - P 223 psi, T 297°F, F 305 gpm 15:30 - P 224 psi,

**#6** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT7, 8-13-2024#p001#c01` (score=0.817)  
30 - P 183 psi, T 322°F, F 240 gpm 09:30 - P 183 psi, T 322°F, F 243 gpm 11:30 - P 182 psi, T 317°F, F 246 gpm 13:30 - P 184 psi, T 318°F, F 251 gpm 15:30 - P 192 psi, T 294°F, F 2

### Hybrid (RRF)

**#1** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT10, 8-16-2024#p001#c02` (score=0.0164 | bm25 #1 | dense #-)  
.0 10,208.1 8,061.0 INT2 9.500 38 OTHER Bulk InventoryWater / Fluids InventoryWater / Fluids HauledSafety Information First Aid Treatments: 0 0 0Medical Treatments: Lost Time Incid

**#2** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT4, 8-9-2024#p001#c01` (score=0.0164 | bm25 #- | dense #1)  
987.0 OTHER Test additional lines, check all flowline valves, shut in 16B(78)-32, change valves on 16A(78)-32 to go downhole and set up sampling ports. 9:00 6:00 21.00 10,987.0 PUM

**#3** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT10, 8-16-2024#p001#c00` (score=0.0161 | bm25 #2 | dense #-)  
10Report No: Report For 06:00 AM 16-Aug-24 Daily Report Well ID: Utah FORGE FORGE 16B(78)-32 Well Name: FORGE 16B(78)-32 Field: FORGE Sect: 32 Town: 26S Rng: 9W County: BEAVER Stat

**#4** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT5, 8-11-2024#p001#c01` (score=0.0161 | bm25 #- | dense #2)  
80 psi, T 278°F, F 121 gpm 13:00 - P 80 psi, T 282°F, F 122 gpm 13:00 6:00 17.00 10,947.0 EVAL Increase pump rate on 16A(78)-32 to 7.5 bpm, Step 6 of pump schedule. 13:30 - P 82 ps

**#5** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT10, 8-15-2024#p001#c01` (score=0.0159 | bm25 #3 | dense #-)  
11:30 - 2,875 psi 13:30 - 2,866 psi 15:30 - 2,925 psi 17:30 - 2,929 psi 19:30 - 2,944 psi 21:30 - 2,931 psi 23:30 - 2,936 psi 01:30 - 2,935 psi 03:30 - lost communication 06:00 - 2

**#6** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT3, 8-9-2024#p001#c01` (score=0.0159 | bm25 #- | dense #3)  
30 Day Circulation Test. 09:15 - 16B wellhead pressure 0 psi, 12:00 - 27 psi, 14:00 - 63 psi, 16:00 - 99 psi, 18:00 - 133 psi, 20:00 - 164 psi, 22:00 - 192 psi, 00:00 - 221 psi, 02

In [5]:
# Try your own question
compare_query("wellhead pressure trend during circulation")

## Query: `wellhead pressure trend during circulation`

### BM25

**#1** `inj_prod/16B/University of Utah_Forge16B(78)-32_SLB_28Aug2024_DiDrill_Production Log Final Report#p004#c00` (score=4.687)  
University of Utah Forge 16B(78)-32 Interpretation Remarks SLB University of Utah / Forge 16B(78)-32 / Producing Profile / Final Report llk 4/14 This interpretation is based on Hi-

**#2** `inj_prod/16B/University of Utah_Forge16B(78)-32_SLB_28Aug2024_DiDrill_Production Log Final Report#p003#c00` (score=4.323)  
University of Utah Forge 16B(78)-32 Objectives: The main objective is to locate downhole water entries and provide a production profile. Well / Job Information: Casing: 7”, 38#, VS

**#3** `inj_prod/16A/University of Utah_Forge16A(78)-32_SLB_17Aug2024_Injection Profile Final Report#p004#c00` (score=3.970)  
University of Utah Forge 16A(78)-32 Interpretation Remarks SLB University of Utah / Forge 16A(78)-32 / Injection Profile / Final Report llk 4/15 This interpretation is based on Ult

**#4** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT4, 8-9-2024#p001#c01` (score=3.641)  
987.0 OTHER Test additional lines, check all flowline valves, shut in 16B(78)-32, change valves on 16A(78)-32 to go downhole and set up sampling ports. 9:00 6:00 21.00 10,987.0 PUM

**#5** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT7, 8-12-2024#p001#c00` (score=3.542)  
7Report No: Report For 06:00 AM 12-Aug-24 Daily Report Well ID: Utah FORGE 16A(78)-32-STIM1 Well Name: FORGE 16A(78)-32-STIM1 Field: FORGE County: Beaver State: UT Country: United 

**#6** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT30, 9-5-2024#p001#c00` (score=3.514)  
30Report No: Report For 06:00 AM 05-Sep-24 Daily Report Well ID: Utah FORGE FORGE 16B(78)-32 Well Name: FORGE 16B(78)-32 Field: FORGE Sect: 32 Town: 26S Rng: 9W County: BEAVER Stat

### Dense (BGE + Chroma)

**#1** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT3, 8-9-2024#p001#c01` (score=0.748)  
30 Day Circulation Test. 09:15 - 16B wellhead pressure 0 psi, 12:00 - 27 psi, 14:00 - 63 psi, 16:00 - 99 psi, 18:00 - 133 psi, 20:00 - 164 psi, 22:00 - 192 psi, 00:00 - 221 psi, 02

**#2** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT7, 8-12-2024#p001#c01` (score=0.737)  
01:30 - 2,822 psi 03:30 - 2,843 psi 05:30 - 2,834 psi Management Summary Increase pump rate to 10.0 bpm, Step 7 on pump schedule. Monitor wellhead pressure. Bulk InventoryWater / F

**#3** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT20, 8-26-2024#p001#c01` (score=0.729)  
24.00 10,947.0 EVAL Continue pumping Step 7 of pump schedule. Monitor wellhead pressure, flowline temperature and separator liquid discharge flow. 07:30 - P 273 psi, T 361°F, F 349

**#4** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT19, 8-25-2024#p001#c01` (score=0.726)  
07:30 - P 271 psi, T 358°F, F 347 gpm 09:30 - P 270 psi, T 355°F, F 348 gpm 11:30 - P 270 psi, T 356°F, F 351 gpm 13:30 - P 270 psi, T 355°F, F 350 gpm 15:30 - P 270 psi, T 355°F, 

**#5** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT14, 8-19-2024#p001#c01` (score=0.723)  
.0 LOG Equalize pressure on lubricator, open well and run in the hole to 2,000' @ 250 fpm. Bring Liberty pumps online at 3 bpm and run in to 5,800'. Slow rate of decent to 66 fpm t

**#6** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT13, 8-19-2024#p001#c01` (score=0.722)  
of pump schedule. Vary rates as necessary while running the EV camera in 16A well. Return to 10 bpm when job complete. Monitor pressure, temperature and flow on 16B. Separator rock

### Hybrid (RRF)

**#1** `inj_prod/16B/University of Utah_Forge16B(78)-32_SLB_28Aug2024_DiDrill_Production Log Final Report#p004#c00` (score=0.0164 | bm25 #1 | dense #-)  
University of Utah Forge 16B(78)-32 Interpretation Remarks SLB University of Utah / Forge 16B(78)-32 / Producing Profile / Final Report llk 4/14 This interpretation is based on Hi-

**#2** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT3, 8-9-2024#p001#c01` (score=0.0164 | bm25 #- | dense #1)  
30 Day Circulation Test. 09:15 - 16B wellhead pressure 0 psi, 12:00 - 27 psi, 14:00 - 63 psi, 16:00 - 99 psi, 18:00 - 133 psi, 20:00 - 164 psi, 22:00 - 192 psi, 00:00 - 221 psi, 02

**#3** `inj_prod/16B/University of Utah_Forge16B(78)-32_SLB_28Aug2024_DiDrill_Production Log Final Report#p003#c00` (score=0.0161 | bm25 #2 | dense #-)  
University of Utah Forge 16B(78)-32 Objectives: The main objective is to locate downhole water entries and provide a production profile. Well / Job Information: Casing: 7”, 38#, VS

**#4** `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT7, 8-12-2024#p001#c01` (score=0.0161 | bm25 #- | dense #2)  
01:30 - 2,822 psi 03:30 - 2,843 psi 05:30 - 2,834 psi Management Summary Increase pump rate to 10.0 bpm, Step 7 on pump schedule. Monitor wellhead pressure. Bulk InventoryWater / F

**#5** `inj_prod/16A/University of Utah_Forge16A(78)-32_SLB_17Aug2024_Injection Profile Final Report#p004#c00` (score=0.0159 | bm25 #3 | dense #-)  
University of Utah Forge 16A(78)-32 Interpretation Remarks SLB University of Utah / Forge 16A(78)-32 / Injection Profile / Final Report llk 4/15 This interpretation is based on Ult

**#6** `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT20, 8-26-2024#p001#c01` (score=0.0159 | bm25 #- | dense #3)  
24.00 10,947.0 EVAL Continue pumping Step 7 of pump schedule. Monitor wellhead pressure, flowline temperature and separator liquid discharge flow. 07:30 - P 273 psi, T 361°F, F 349